In [1]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

anthropic_client = Anthropic(
    api_key=os.getenv("ANTHROPIC_API_KEY")
)

def llm(prompt):
    response = anthropic_client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=1000,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.content[0].text

In [2]:
question = "I just discovered the course. Can I join now?"

answer = llm(question)
print(answer)

I'd be happy to help you join! However, I need a bit more information to give you the best answer:

1. **Which course** are you interested in joining?
2. **When does it start** (or when did it start)?
3. **What platform or institution** is offering it?

In general:
- **Online courses** (like Coursera, edX, Udemy) often allow late enrollment or self-paced learning
- **University/college courses** may have add/drop deadlines (usually within the first 1-2 weeks)
- **Live cohort-based courses** might have stricter start dates

**What you can do right now:**
- Check the course website for enrollment deadlines
- Contact the instructor or course administrator directly
- Look for a "Join" or "Enroll" button to see if registration is still open

If you share more details about the specific course, I can give you more targeted advice!


# RAG version

In [4]:
def build_prompt(question, search_results):
    context = ""

    for doc in search_results:
        context = context + f"Question: {doc['question']}\n"
        context = context + f"Answer: {doc['text']}\n\n"

    prompt = f"""
Your task is to answer questions from course participants
based on the provided context.

Use only the context to answer the question.
If the answer is not found in the context, say "I don't know."

Question:
{question}

Context:
{context}
""".strip()

    return prompt


def rag(question):
    search_results = search(question)
    prompt = build_prompt(question, search_results)
    answer = llm(prompt)
    return answer

In [5]:
llm("Hey, what's up?")

"Hey! Not much, just here and ready to chat. What's on your mind?"

In [6]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

I'd be happy to help you join! However, I need a bit more information to give you the best answer:

1. **Which course** are you interested in joining?
2. **When does it start** (or when did it start)?
3. **Where did you find it** - is this through a school, online platform, or other organization?

Many courses allow late registration, especially:
- Online self-paced courses (you can usually join anytime)
- Courses in their first week or two
- Some programs with rolling admissions

If you let me know the specific course details, I can help you figure out if late enrollment is possible and what steps you'd need to take!


In [7]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [8]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [9]:
answer = llm(prompt)
print(answer)

Based on the context provided, **yes, you can still join the course now**. 

However, there's an important note: if you want to receive a certificate, you need to submit your project while submissions are still being accepted.

You can start learning and submitting homework right away without needing any special confirmation - registration is just to gauge interest, and it's not checked against any registered list.


In [10]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [11]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)


1349

In [12]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [13]:
from minsearch import Index

index = Index(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course']
)
index.fit(documents)

In [14]:


search_results = index.search(
    question,
    boost_dict={'question': 2.0, 'section': 0.5},
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

search_results



[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [15]:


def search(question, course='llm-zoomcamp'):
    boost_dict = {'question': 2.0, 'section': 0.5}
    filter_dict = {'course': course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )



In [16]:
search_results = search(question)

In [18]:
USER_PROMPT_TEMPALATE = '''
Question:
{question}

Context:
{context}
'''



In [19]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [20]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc['section'])
        lines.append('Q: ' + doc['question'])
        lines.append('A: ' + doc['answer'])
        lines.append('')

    return '\n'.join(lines).strip()

In [25]:

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPALATE.format(
        question=question,
        context=context
    )
    return prompt.strip()



In [26]:

prompt = build_prompt(question, search_results)

